# Low-rank 근사와 이미지 압축

> 선형대수 17강 · 특이값 분해

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [Low-rank 근사와 이미지 압축](https://mioon1402.github.io/timeseriesdata/linalg/L17-lowrank.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 0. 기초 다지기 — 랭크 1 행렬이란

## 1. SVD를 층으로 쌓기

## 2. 직접 압축해보기

## 3. Eckart-Young 정리

## 4. 얼마나 이득인가

## 5. 잡음 제거 — 왜 되는가

## 6. numpy 로 확인하기

**17-1. 이미지 만들고 SVD**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def 그림(n=64):
    y, x = np.mgrid[0:n, 0:n] / n
    img = 0.5 + 0.4 * np.sin(6 * np.pi * x) * np.cos(4 * np.pi * y)
    img += 0.35 * ((x - 0.3) ** 2 + (y - 0.65) ** 2 < 0.03)     # 동그라미
    img[int(.2 * n):int(.3 * n), int(.1 * n):int(.9 * n)] = 0.05  # 가로 막대
    return np.clip(img, 0, 1)

img = 그림()
U, σ, Vt = np.linalg.svd(img)

print("이미지 크기:", img.shape, " 랭크:", np.linalg.matrix_rank(img))
print("특이값 상위 8개:", np.round(σ[:8], 3))
print("특이값 하위 8개:", np.round(σ[-8:], 6))
print()
누적 = np.cumsum(σ ** 2) / np.sum(σ ** 2)
for k in [1, 3, 5, 10, 20]:
    print(f"  상위 {k:2d}개가 담은 에너지: {누적[k-1]:.2%}")

**17-2. 랭크 k 근사와 오차**

In [ ]:
def 근사(U, σ, Vt, k):
    return U[:, :k] @ np.diag(σ[:k]) @ Vt[:k]

m, n = img.shape
원본크기 = m * n

print(f"{'k':>4} {'상대오차':>10} {'이론값':>10} {'저장':>8} {'압축률':>8}")
for k in [1, 2, 3, 5, 10, 20, 40]:
    Ak = 근사(U, σ, Vt, k)
    실제 = np.linalg.norm(img - Ak) / np.linalg.norm(img)
    이론 = np.sqrt((σ[k:] ** 2).sum()) / np.linalg.norm(img)   # 버린 σ 들만으로
    저장 = k * (m + n + 1)
    print(f"{k:>4} {실제:>10.2%} {이론:>10.2%} {저장:>8} {저장/원본크기:>8.1%}")

print()
print("→ '실제 오차' 와 '버린 σ 로 계산한 이론값' 이 정확히 일치한다.")
print("   Eckart-Young 정리 덕분에 오차를 미리 알 수 있다.")

**17-3. 정말 최선인가 — 무작위 랭크 k 와 비교**

In [ ]:
rng = np.random.default_rng(0)
k = 5
svd_근사 = 근사(U, σ, Vt, k)
svd_오차 = np.linalg.norm(img - svd_근사)

print(f"SVD 랭크 {k} 근사 오차: {svd_오차:.4f}")
print()
print("무작위 랭크 5 행렬 중 가장 좋았던 것:")
최선 = np.inf
for _ in range(2000):
    Bu = rng.normal(size=(64, k))
    Bv = rng.normal(size=(k, 64))
    B = Bu @ Bv
    B = B * (np.sum(img * B) / np.sum(B * B))     # 크기만 최적으로 맞춰줌
    최선 = min(최선, np.linalg.norm(img - B))
print(f"  {최선:.4f}   ← SVD 보다 {최선/svd_오차:.1f}배 나쁘다")
print()
print("→ 어떤 랭크 5 행렬도 SVD 절단을 이길 수 없다 (Eckart-Young)")

**17-4. 눈으로 보기**

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, k in zip(axes, [1, 3, 5, 15, None]):
    보여줄 = img if k is None else 근사(U, σ, Vt, k)
    ax.imshow(보여줄, cmap="gray", vmin=0, vmax=1)
    오차 = 0 if k is None else np.linalg.norm(img - 보여줄) / np.linalg.norm(img)
    ax.set_title("원본" if k is None else f"k={k}\n오차 {오차:.1%}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

**17-5. 특이값 스펙트럼**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(σ, "o-", ms=3)
axes[0].set_yscale("log")
axes[0].set_title("특이값 (로그 눈금)")
axes[0].set_xlabel("번호")

누적 = np.cumsum(σ ** 2) / np.sum(σ ** 2)
axes[1].plot(np.arange(1, len(σ) + 1), 누적, "o-", ms=3)
axes[1].axhline(0.99, ls="--", c="red", lw=1)
axes[1].set_title("누적 에너지 비율")
axes[1].set_xlabel("남긴 개수 k")
axes[1].set_xlim(0, 30)

plt.tight_layout()
plt.show()

k99 = int(np.searchsorted(누적, 0.99)) + 1
print(f"에너지의 99% 를 담으려면 k = {k99} 개면 충분하다")

**17-6. 잡음 제거 — 적당한 k 가 최선**

In [ ]:
rng = np.random.default_rng(0)
잡음본 = img + rng.normal(0, 0.12, img.shape)

Un, σn, Vtn = np.linalg.svd(잡음본)
기준 = np.linalg.norm(img - 잡음본) / np.linalg.norm(img)

print(f"잡음본 자체의 오차: {기준:.3f}")
print()
print(f"{'k':>4} {'원본과의 오차':>14}")
for k in [2, 5, 8, 12, 20, 40, 64]:
    복원 = Un[:, :k] @ np.diag(σn[:k]) @ Vtn[:k]
    오차 = np.linalg.norm(img - 복원) / np.linalg.norm(img)
    표시 = "  ← 가장 좋음" if 오차 < 기준 * 0.55 else ""
    print(f"{k:>4} {오차:>14.3f}{표시}")

print()
print("→ k 가 작으면 구조까지 잃고, 크면 잡음까지 되살아난다.")
print("   중간 어딘가가 최선 — 머신러닝의 과적합과 같은 구조다.")

**17-7. 연습문제**

In [ ]:
# 문제 1. 랭크가 정확히 3인 행렬을 만들고, k=3 근사의 오차가 0인지 확인하세요.
#         (힌트: 랭크1 세 개를 더하면 됩니다)

# 문제 2. 위 그림에서 오차를 5% 이하로 하려면 k 가 몇이어야 할까요?
#         σ 만 보고 미리 계산해보세요.

# 문제 3. 무작위 행렬(구조가 없는)의 특이값은 어떤 모양일까요?
#         압축이 잘 될까요?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
rng2 = np.random.default_rng(7)

# 문제 1
R = sum(np.outer(rng2.normal(size=20), rng2.normal(size=30)) for _ in range(3))
U1, s1, V1 = np.linalg.svd(R)
print("문제 1: 랭크 =", np.linalg.matrix_rank(R))
print("        σ =", np.round(s1[:5], 6), " ← 4번째부터 0")
A3 = U1[:, :3] @ np.diag(s1[:3]) @ V1[:3]
print("        k=3 오차 =", np.linalg.norm(R - A3))

# 문제 2 — 버린 σ 로 미리 계산
전체 = np.linalg.norm(σ)
for k in range(1, 30):
    if np.sqrt((σ[k:]**2).sum()) / 전체 <= 0.05:
        print(f"\n문제 2: k = {k} 면 오차 {np.sqrt((σ[k:]**2).sum())/전체:.2%}")
        break

# 문제 3 — 무작위는 σ 가 완만하게 떨어져 압축이 안 된다
노이즈 = rng2.normal(size=(64, 64))
sn2 = np.linalg.svd(노이즈, compute_uv=False)
누적n = np.cumsum(sn2**2) / np.sum(sn2**2)
print(f"\n문제 3: 무작위 행렬 — 에너지 99% 에 필요한 k = {int(np.searchsorted(누적n, 0.99))+1} / 64")
print(f"        그림 — 필요한 k = {int(np.searchsorted(np.cumsum(σ**2)/np.sum(σ**2), 0.99))+1} / 64")
print("→ 구조가 없으면 압축할 게 없다. 압축률이 곧 '구조의 양' 이다.")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)